In [1]:
import json
from pathlib import Path

# Paths
dataset_dir = Path(r"d:\RecSys\Yelp-JSON\Yelp JSON\yelp_dataset")
business_file = dataset_dir / "yelp_academic_dataset_business.json"
review_file = dataset_dir / "yelp_academic_dataset_review.json"

# Output files
out_business_ids = Path("ouputs/philadelphia_business_ids.txt")
out_user_ids = Path("ouputs/philadelphia_user_ids.txt")

TARGET_CITY = "Philadelphia"

# 1) Collect all business IDs in Philadelphia
print(f"[1/4] Scanning businesses for city='{TARGET_CITY}' ...")
philly_business_ids = set()
business_line_count = 0
with business_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        business_line_count += 1
        obj = json.loads(line)
        if obj.get("city") == TARGET_CITY:
            bid = obj.get("business_id")
            if bid:
                philly_business_ids.add(bid)
print(f"    Scanned {business_line_count:,} business records → found {len(philly_business_ids):,} in {TARGET_CITY}")

# Save business IDs (one per line)
print(f"[2/4] Saving business IDs to '{out_business_ids}' ...")
out_business_ids.parent.mkdir(parents=True, exist_ok=True)
with out_business_ids.open("w", encoding="utf-8") as f:
    for bid in sorted(philly_business_ids):
        f.write(bid + "\n")
print(f"    Saved {len(philly_business_ids):,} business IDs.")

# 2) Collect all users who reviewed those Philadelphia businesses
print(f"[3/4] Scanning reviews for {TARGET_CITY} businesses ...")
philly_user_ids = set()
review_line_count = 0
matched_reviews = 0
with review_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        review_line_count += 1
        if review_line_count % 500_000 == 0:
            print(f"    ... processed {review_line_count:,} reviews so far (users collected: {len(philly_user_ids):,})")
        obj = json.loads(line)
        if obj.get("business_id") in philly_business_ids:
            matched_reviews += 1
            uid = obj.get("user_id")
            if uid:
                philly_user_ids.add(uid)
print(f"    Scanned {review_line_count:,} review records → {matched_reviews:,} matched → {len(philly_user_ids):,} unique users")

# Save user IDs (one per line)
print(f"[4/4] Saving user IDs to '{out_user_ids}' ...")
out_user_ids.parent.mkdir(parents=True, exist_ok=True)
with out_user_ids.open("w", encoding="utf-8") as f:
    for uid in sorted(philly_user_ids):
        f.write(uid + "\n")
print(f"    Saved {len(philly_user_ids):,} user IDs.")

print("\nDone")
print(f"Philadelphia business IDs: {len(philly_business_ids):,}")
print(f"Users who reviewed them:   {len(philly_user_ids):,}")
print(f"Saved business IDs to: {out_business_ids}")
print(f"Saved user IDs to:     {out_user_ids}")


[1/4] Scanning businesses for city='Philadelphia' ...
    Scanned 150,346 business records → found 14,569 in Philadelphia
[2/4] Saving business IDs to 'ouputs\philadelphia_business_ids.txt' ...
    Saved 14,569 business IDs.
[3/4] Scanning reviews for Philadelphia businesses ...
    ... processed 500,000 reviews so far (users collected: 46,684)
    ... processed 1,000,000 reviews so far (users collected: 77,901)
    ... processed 1,500,000 reviews so far (users collected: 104,517)
    ... processed 2,000,000 reviews so far (users collected: 128,322)
    ... processed 2,500,000 reviews so far (users collected: 150,302)
    ... processed 3,000,000 reviews so far (users collected: 169,612)
    ... processed 3,500,000 reviews so far (users collected: 186,539)
    ... processed 4,000,000 reviews so far (users collected: 201,894)
    ... processed 4,500,000 reviews so far (users collected: 215,971)
    ... processed 5,000,000 reviews so far (users collected: 229,776)
    ... processed 5,500,

In [3]:
import json
from pathlib import Path

# Input files
business_ids_path = Path("ouputs/philadelphia_business_ids.txt")
user_ids_path = Path("ouputs/philadelphia_user_ids.txt")
review_file = dataset_dir / "yelp_academic_dataset_review.json"

# Output file
out_edges_user_business = Path("ouputs/edges_user_business.txt")


def load_id_file(path: Path) -> set[str]:
    ids = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            value = line.strip()
            if value:
                ids.add(value)
    return ids


print("[1/5] Loading business IDs ...")
business_ids = load_id_file(business_ids_path)
print(f"    Loaded {len(business_ids):,} business IDs from {business_ids_path}")

print("[2/5] Loading user IDs ...")
user_ids = load_id_file(user_ids_path)
print(f"    Loaded {len(user_ids):,} user IDs from {user_ids_path}")

print("[3/5] Building user-business edges from reviews (stars >= 4) ...")
edge_set = set()
review_count = 0
candidate_review_count = 0
skipped_low_rating = 0
skipped_outside_filter = 0

with review_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        review_count += 1
        if review_count % 500_000 == 0:
            print(
                f"    ... processed {review_count:,} reviews | "
                f"candidate positives: {candidate_review_count:,} | "
                f"unique edges: {len(edge_set):,}"
            )

        obj = json.loads(line)
        business_id = obj.get("business_id")
        user_id = obj.get("user_id")
        stars = obj.get("stars", 0)

        if business_id not in business_ids or user_id not in user_ids:
            skipped_outside_filter += 1
            continue

        if stars < 4:
            skipped_low_rating += 1
            continue

        candidate_review_count += 1
        edge_set.add((user_id, business_id))

print(f"    Finished scanning {review_count:,} reviews")
print(f"    Filtered-out reviews (user/business not in ID files): {skipped_outside_filter:,}")
print(f"    Filtered-out reviews (stars < 4): {skipped_low_rating:,}")
print(f"    Positive reviews after filtering: {candidate_review_count:,}")
print(f"    Unique user-business edges: {len(edge_set):,}")

print(f"[4/5] Saving edges to {out_edges_user_business} ...")
out_edges_user_business.parent.mkdir(parents=True, exist_ok=True)
with out_edges_user_business.open("w", encoding="utf-8") as f:
    for user_id, business_id in sorted(edge_set):
        f.write(f"{user_id}\t{business_id}\n")

edge_users = {user_id for user_id, _ in edge_set}
edge_businesses = {business_id for _, business_id in edge_set}

print("[5/5] Summary")
print(f"    Businesses in ID file: {len(business_ids):,}")
print(f"    Users in ID file:      {len(user_ids):,}")
print(f"    Businesses in edges:   {len(edge_businesses):,}")
print(f"    Users in edges:        {len(edge_users):,}")
print(f"    Total edges:           {len(edge_set):,}")
print(f"    Saved to:              {out_edges_user_business}")


[1/5] Loading business IDs ...
    Loaded 14,569 business IDs from ouputs\philadelphia_business_ids.txt
[2/5] Loading user IDs ...
    Loaded 279,857 user IDs from ouputs\philadelphia_user_ids.txt
[3/5] Building user-business edges from reviews (stars >= 4) ...
    ... processed 500,000 reviews | candidate positives: 53,433 | unique edges: 52,215
    ... processed 1,000,000 reviews | candidate positives: 104,568 | unique edges: 102,075
    ... processed 1,500,000 reviews | candidate positives: 150,788 | unique edges: 147,009
    ... processed 2,000,000 reviews | candidate positives: 200,311 | unique edges: 195,353
    ... processed 2,500,000 reviews | candidate positives: 250,692 | unique edges: 244,517
    ... processed 3,000,000 reviews | candidate positives: 295,090 | unique edges: 287,725
    ... processed 3,500,000 reviews | candidate positives: 339,082 | unique edges: 330,279
    ... processed 4,000,000 reviews | candidate positives: 385,482 | unique edges: 375,634
    ... proces

In [8]:
import json
from pathlib import Path

# Input files
user_file = dataset_dir / "yelp_academic_dataset_user.json"
review_file = dataset_dir / "yelp_academic_dataset_review.json"
positive_edge_path = Path("ouputs/edges_user_business.txt")
business_ids_path = Path("ouputs/philadelphia_business_ids.txt")
TOP_K_FRIENDS = 15
FRIEND_MIN_STARS = 3

# Output file
out_edges_user_user = Path("ouputs/edges_user_user.txt")


def load_positive_users_from_edge_file(path: Path) -> set[str]:
    user_ids = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            value = line.strip()
            if not value:
                continue
            user_id, _ = value.split("\t", 1)
            user_ids.add(user_id)
    return user_ids


def load_id_file(path: Path) -> set[str]:
    ids = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            value = line.strip()
            if value:
                ids.add(value)
    return ids


def load_friend_eligible_users(review_path: Path, business_ids: set[str], min_stars: float) -> set[str]:
    eligible_users = set()
    review_count = 0
    matched_business_reviews = 0

    with review_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            review_count += 1
            if review_count % 500_000 == 0:
                print(
                    f"    ... processed {review_count:,} reviews for friend filter | "
                    f"eligible friends: {len(eligible_users):,}"
                )

            obj = json.loads(line)
            if obj.get("business_id") not in business_ids:
                continue

            matched_business_reviews += 1
            if obj.get("stars", 0) >= min_stars:
                user_id = obj.get("user_id")
                if user_id:
                    eligible_users.add(user_id)

    print(f"    Reviews scanned for friend filter: {review_count:,}")
    print(f"    Reviews on Philadelphia businesses: {matched_business_reviews:,}")
    print(f"    Eligible friend users (Philadelphia reviews with stars >= {min_stars}): {len(eligible_users):,}")
    return eligible_users


print("[1/5] Loading user and business filters ...")
positive_user_ids = load_positive_users_from_edge_file(positive_edge_path)
philly_business_ids = load_id_file(business_ids_path)
print(f"    Source users with positive interaction (stars >= 4): {len(positive_user_ids):,}")
print(f"    Philadelphia businesses: {len(philly_business_ids):,}")
print(f"    Using top-{TOP_K_FRIENDS} friends per user")
print(f"    Friend condition: reviewed a Philadelphia business with stars >= {FRIEND_MIN_STARS}")

print("[2/5] Building eligible friend user set from reviews ...")
friend_eligible_user_ids = load_friend_eligible_users(
    review_file,
    philly_business_ids,
    FRIEND_MIN_STARS,
)

print("[3/5] Scanning user social graph ...")
edge_set = set()
processed_users = 0
source_users_in_scope = 0
source_users_with_friends = 0
friend_links_seen = 0
friend_links_kept = 0

with user_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        processed_users += 1
        if processed_users % 200_000 == 0:
            print(
                f"    ... processed {processed_users:,} user records | "
                f"positive source users in scope: {source_users_in_scope:,} | "
                f"kept edges: {len(edge_set):,}"
            )

        obj = json.loads(line)
        user_id = obj.get("user_id")
        if user_id not in positive_user_ids:
            continue

        source_users_in_scope += 1
        friends_raw = obj.get("friends", "")
        if not friends_raw or friends_raw == "None":
            continue

        friend_list = [friend.strip() for friend in friends_raw.split(",") if friend.strip()]
        if not friend_list:
            continue

        friend_list = friend_list[:TOP_K_FRIENDS]
        source_users_with_friends += 1
        for friend_id in friend_list:
            friend_links_seen += 1
            if friend_id in friend_eligible_user_ids:
                edge_set.add((user_id, friend_id))
                friend_links_kept += 1

print(f"    Finished scanning {processed_users:,} user records")
print(f"    Positive source users found in user dataset: {source_users_in_scope:,}")
print(f"    Positive source users with friend list: {source_users_with_friends:,}")
print(f"    Friend links examined after top-{TOP_K_FRIENDS} cap: {friend_links_seen:,}")
print(f"    Friend links where friend passed the Philadelphia rating filter: {friend_links_kept:,}")
print(f"    Unique user-user edges: {len(edge_set):,}")

print(f"[4/5] Saving user-user edges to {out_edges_user_user} ...")
out_edges_user_user.parent.mkdir(parents=True, exist_ok=True)
with out_edges_user_user.open("w", encoding="utf-8") as f:
    for user_id, friend_id in sorted(edge_set):
        f.write(f"{user_id}\t{friend_id}\n")

users_as_source = {user_id for user_id, _ in edge_set}
users_as_target = {friend_id for _, friend_id in edge_set}
users_in_edges = users_as_source | users_as_target

print("[5/5] Summary")
print(f"    Positive source users:         {len(positive_user_ids):,}")
print(f"    Eligible Philadelphia friends: {len(friend_eligible_user_ids):,}")
print(f"    Users as edge source:          {len(users_as_source):,}")
print(f"    Users as edge target:          {len(users_as_target):,}")
print(f"    Distinct users in edge:        {len(users_in_edges):,}")
print(f"    Total edges:                   {len(edge_set):,}")
print(f"    Saved to:                      {out_edges_user_user}")


[1/5] Loading user and business filters ...
    Source users with positive interaction (stars >= 4): 208,801
    Philadelphia businesses: 14,569
    Using top-15 friends per user
    Friend condition: reviewed a Philadelphia business with stars >= 3
[2/5] Building eligible friend user set from reviews ...
    ... processed 500,000 reviews for friend filter | eligible friends: 38,283
    ... processed 1,000,000 reviews for friend filter | eligible friends: 63,463
    ... processed 1,500,000 reviews for friend filter | eligible friends: 84,143
    ... processed 2,000,000 reviews for friend filter | eligible friends: 103,504
    ... processed 2,500,000 reviews for friend filter | eligible friends: 120,304
    ... processed 3,000,000 reviews for friend filter | eligible friends: 134,628
    ... processed 3,500,000 reviews for friend filter | eligible friends: 147,820
    ... processed 4,000,000 reviews for friend filter | eligible friends: 159,932
    ... processed 4,500,000 reviews for fr

In [2]:
import pickle
from pathlib import Path

emb_path = Path(r"d:\RecSys\srcs\business_embeddings.pkl")
philly_ids_path = Path("ouputs/philadelphia_business_ids.txt")

# Load embeddings
with emb_path.open("rb") as f:
    embeddings = pickle.load(f)

# Load Philadelphia business IDs
philly_ids = set()
with philly_ids_path.open("r", encoding="utf-8") as f:
    for line in f:
        v = line.strip()
        if v:
            philly_ids.add(v)

# Detect structure
if isinstance(embeddings, dict):
    emb_keys = set(embeddings.keys())
    emb_count = len(emb_keys)
elif hasattr(embeddings, "__len__"):
    emb_keys = None
    emb_count = len(embeddings)
else:
    emb_keys = None
    emb_count = None

print("=== Embedding file info ===")
print(f"Type:            {type(embeddings)}")
if hasattr(embeddings, "shape"):
    print(f"Shape:           {embeddings.shape}")
print(f"Total entries:   {emb_count:,}" if emb_count is not None else "Total entries: unknown")

if emb_keys is not None:
    first_val = next(iter(embeddings.values()))
    print(f"Key sample:      {list(emb_keys)[:2]}")
    print(f"Value type:      {type(first_val)}")
    if hasattr(first_val, "shape"):
        print(f"Embedding dim:   {first_val.shape}")

print()
print("=== Cross-check with Philadelphia IDs ===")
print(f"Philadelphia business IDs:   {len(philly_ids):,}")
print(f"Embedding entries:           {emb_count:,}" if emb_count is not None else "Embedding entries: unknown")

if emb_keys is not None:
    in_both = philly_ids & emb_keys
    only_in_philly = philly_ids - emb_keys
    only_in_emb = emb_keys - philly_ids
    print(f"Matched (in both):           {len(in_both):,}")
    print(f"Missing from embedding:      {len(only_in_philly):,}  ← Philadelphia IDs without embedding")
    print(f"Extra in embedding:          {len(only_in_emb):,}  ← embeddings not in Philadelphia IDs")
    if len(in_both) == len(philly_ids) == emb_count:
        print("\n✔ PERFECT MATCH — embedding count equals Philadelphia business count")
    else:
        print("\n✘ MISMATCH — see counts above for detail")


=== Embedding file info ===
Type:            <class 'dict'>
Total entries:   14,569
Key sample:      ['1DFvkUjtsqNRNlORaUYlFQ', '3DFJuwPDIPbEUwA6Tp9E6Q']
Value type:      <class 'numpy.ndarray'>
Embedding dim:   (384,)

=== Cross-check with Philadelphia IDs ===
Philadelphia business IDs:   14,569
Embedding entries:           14,569
Matched (in both):           14,569
Missing from embedding:      0  ← Philadelphia IDs without embedding
Extra in embedding:          0  ← embeddings not in Philadelphia IDs

✔ PERFECT MATCH — embedding count equals Philadelphia business count


In [4]:
import json
import random
from pathlib import Path

# Paths
edges_path  = Path("ouputs/edges_business_business_similar.txt")
biz_file    = Path(r"d:\RecSys\Yelp-JSON\Yelp JSON\yelp_dataset\yelp_academic_dataset_business.json")
philly_ids_path = Path("ouputs/philadelphia_business_ids.txt")
SAMPLE_N    = 3   # số cặp muốn xem

# 1. Load Philadelphia IDs
philly_ids = set()
with philly_ids_path.open("r", encoding="utf-8") as f:
    for line in f:
        v = line.strip()
        if v:
            philly_ids.add(v)

# 2. Load business metadata
print("Loading business metadata ...")
meta = {}
with biz_file.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        bid = obj.get("business_id")
        if bid in philly_ids:
            meta[bid] = {
                "name":       obj.get("name", "N/A"),
                "categories": obj.get("categories") or "N/A",
                "stars":      obj.get("stars", "N/A"),
                "address":    obj.get("address", ""),
            }
print(f"Loaded metadata for {len(meta):,} businesses\n")

# 3. Load all edges, sample N pairs
all_edges = []
with edges_path.open("r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")
        if len(parts) == 2:
            all_edges.append((parts[0], parts[1]))

print(f"Total edges in file: {len(all_edges):,}")
sample_edges = random.sample(all_edges, min(SAMPLE_N, len(all_edges)))

# 4. In thông tin từng cặp
print(f"\n=== {SAMPLE_N} sample business-business similar pairs ===\n")
for idx, (bid_a, bid_b) in enumerate(sample_edges, 1):
    ma = meta.get(bid_a, {})
    mb = meta.get(bid_b, {})
    diff = abs(float(ma.get("stars", 0)) - float(mb.get("stars", 0)))
    print(f"--- Pair {idx} ---")
    print(f"  A  name      : {ma.get('name')}")
    print(f"     categories: {ma.get('categories')}")
    print(f"     stars     : {ma.get('stars')}")
    print(f"     address   : {ma.get('address')}")
    print()
    print(f"  B  name      : {mb.get('name')}")
    print(f"     categories: {mb.get('categories')}")
    print(f"     stars     : {mb.get('stars')}")
    print(f"     address   : {mb.get('address')}")
    print()
    print(f"     |rating_diff| = {diff:.1f}")
    print()


Loading business metadata ...
Loaded metadata for 14,569 businesses

Total edges in file: 171,298

=== 3 sample business-business similar pairs ===

--- Pair 1 ---
  A  name      : Neighborhood Ramen
     categories: Ramen, Japanese, Restaurants
     stars     : 4.0
     address   : 617 S 3rd St

  B  name      : Ai Ramen
     categories: Restaurants, Japanese, Ramen
     stars     : 3.0
     address   : 1625 Chestnut St

     |rating_diff| = 1.0

--- Pair 2 ---
  A  name      : Hand & Stone Massage And Facial Spa
     categories: Day Spas, Beauty & Spas, Skin Care, Massage
     stars     : 3.0
     address   : 1100 S Columbus Blvd, Ste 24

  B  name      : reYOU massage
     categories: Massage, Health & Medical, Beauty & Spas, Medical Spas
     stars     : 4.0
     address   : 1315 Walnut St

     |rating_diff| = 1.0

--- Pair 3 ---
  A  name      : Overbrook Pizza Shop
     categories: Pizza, Restaurants
     stars     : 4.0
     address   : 2099 N 63rd St

  B  name      : Rosa's F

In [5]:
import torch
from torch_geometric.data import HeteroData
from pathlib import Path
import time

# ── Paths ────────────────────────────────────────────────────────────────────
UB_PATH  = Path("ouputs/edges_user_business.txt")       # user ─rates→ business
UU_PATH  = Path("ouputs/edges_user_user.txt")           # user ─friends→ user
BB_PATH  = Path("ouputs/edges_business_business_similar.txt")  # biz ─similar→ biz
OUT_PATH = Path("ouputs/graph.pt")

t0 = time.time()
SEP = "=" * 60

# ─────────────────────────────────────────────────────────────────────────────
# 1. Load raw edges
# ─────────────────────────────────────────────────────────────────────────────
print(SEP)
print("STEP 1 — Loading raw edges")
print(SEP)

def load_edges(path):
    edges = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                edges.append((parts[0], parts[1]))
    return edges

ub_raw = load_edges(UB_PATH)
uu_raw = load_edges(UU_PATH)
bb_raw = load_edges(BB_PATH)

print(f"  user-business edges : {len(ub_raw):>10,}")
print(f"  user-user edges     : {len(uu_raw):>10,}")
print(f"  biz-biz similar     : {len(bb_raw):>10,}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Build ID → integer index mappings
# ─────────────────────────────────────────────────────────────────────────────
print()
print(SEP)
print("STEP 2 — Building node ID → integer index mappings")
print(SEP)

# Collect all unique IDs
all_user_ids = set()
all_biz_ids  = set()

for u, b in ub_raw:
    all_user_ids.add(u)
    all_biz_ids.add(b)

for u, v in uu_raw:
    all_user_ids.add(u)
    all_user_ids.add(v)

for b1, b2 in bb_raw:
    all_biz_ids.add(b1)
    all_biz_ids.add(b2)

# Stable sorted mappings for reproducibility
user2idx = {uid: i for i, uid in enumerate(sorted(all_user_ids))}
biz2idx  = {bid: i for i, bid in enumerate(sorted(all_biz_ids))}

num_users = len(user2idx)
num_biz   = len(biz2idx)

print(f"  Unique users     : {num_users:>10,}")
print(f"  Unique businesses: {num_biz:>10,}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. Convert to edge_index tensors
# ─────────────────────────────────────────────────────────────────────────────
print()
print(SEP)
print("STEP 3 — Converting edges to index tensors")
print(SEP)

def to_edge_index(raw_edges, src_map, dst_map):
    src_list, dst_list = [], []
    skipped = 0
    for s, d in raw_edges:
        if s in src_map and d in dst_map:
            src_list.append(src_map[s])
            dst_list.append(dst_map[d])
        else:
            skipped += 1
    edge_index = torch.tensor([src_list, dst_list], dtype=torch.long)
    return edge_index, skipped

ei_ub, skip_ub = to_edge_index(ub_raw, user2idx, biz2idx)
ei_uu, skip_uu = to_edge_index(uu_raw, user2idx, user2idx)
ei_bb, skip_bb = to_edge_index(bb_raw, biz2idx, biz2idx)

print(f"  (user, rates, business)   shape={list(ei_ub.shape)}  skipped={skip_ub}")
print(f"  (user, friends, user)     shape={list(ei_uu.shape)}  skipped={skip_uu}")
print(f"  (business, similar, biz)  shape={list(ei_bb.shape)}  skipped={skip_bb}")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Build HeteroData
# ─────────────────────────────────────────────────────────────────────────────
print()
print(SEP)
print("STEP 4 — Assembling HeteroData")
print(SEP)

data = HeteroData()

# Node counts (placeholder feature = identity index)
data["user"].num_nodes    = num_users
data["business"].num_nodes = num_biz

# Edge relations
data["user", "rates",   "business"].edge_index = ei_ub
data["user", "friends", "user"    ].edge_index = ei_uu
data["business", "similar", "business"].edge_index = ei_bb

# Reverse edges for undirected message passing
data["business", "rated_by", "user"    ].edge_index = ei_ub.flip(0)
data["business", "similar_rev", "business"].edge_index = ei_bb.flip(0)

print(data)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Save
# ─────────────────────────────────────────────────────────────────────────────
print()
print(SEP)
print("STEP 5 — Saving graph")
print(SEP)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "data":     data,
        "user2idx": user2idx,
        "biz2idx":  biz2idx,
    },
    OUT_PATH,
)
size_mb = OUT_PATH.stat().st_size / (1024 ** 2)
elapsed = time.time() - t0

print(f"  Saved  → {OUT_PATH}")
print(f"  Size   : {size_mb:.2f} MB")
print(f"  Elapsed: {elapsed:.1f}s")
print()
print("✓ Graph build complete.")


ModuleNotFoundError: No module named 'torch'

In [6]:
!nvidia-smi

Tue Mar 31 07:47:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.95                 Driver Version: 581.95         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   48C    P3             18W /   30W |       0MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----